In [1]:
from glob import glob
import shutil
import os

import cv2
import numpy as np
import pandas as pd

In [6]:
phase = 'train'

## Copy files

In [7]:
png_files = glob(f'raw/{phase}/inputs/*.png')
png_files = sorted(png_files, key=lambda x: int(''.join(filter(str.isdigit, x))))

for i, file in enumerate(png_files):
    if file.endswith('.png'):
        # Open the PNG file
        png_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)

        # Create a white background image with the same size as the PNG image
        white_background = 255 * np.ones_like(png_image[..., :3])  # Create a white RGB image

        # Check if the PNG image has an alpha channel
        if png_image.shape[-1] == 4:
            # Merge the RGB channels of the PNG image with the white background
            alpha = png_image[..., 3] / 255.0  # Normalize alpha channel
            white_background = (1.0 - alpha[..., None]) * white_background + alpha[..., None] * png_image[..., :3]

        dst_file = f'processed/{phase}/inputs/{i+1:04d}.bmp'
        cv2.imwrite(dst_file, white_background)

print('Conversion complete.')

Conversion complete.


In [8]:
folders = glob(f'raw/{phase}/targets/**')

i = 1

for folder in folders:
    
    png_files = glob(f'{folder}/*.png')
    png_files = sorted(png_files, key=lambda x: int(''.join(filter(str.isdigit, x))))
    
    for file in png_files:
        if file.endswith('.png'):
            # Open the PNG file
            png_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)

            # Create a white background image with the same size as the PNG image
            white_background = 255 * np.ones_like(png_image[..., :3])  # Create a white RGB image

            # Check if the PNG image has an alpha channel
            if png_image.shape[-1] == 4:
                # Merge the RGB channels of the PNG image with the white background
                alpha = png_image[..., 3] / 255.0  # Normalize alpha channel
                white_background = (1.0 - alpha[..., None]) * white_background + alpha[..., None] * png_image[..., :3]

            dst_file = f'processed/{phase}/targets/{i:04d}.bmp'
            cv2.imwrite(dst_file, white_background)

            i += 1

print('Conversion complete.')

Conversion complete.


## Create a csv file

In [9]:
n = len(os.listdir(f'processed/{phase}/targets/'))
m = 25
colors = ['blue', 'brown', 'red', 'yellow', 'green']
c = len(colors)

# input column
inputs = []
subjects = os.listdir(f'processed/{phase}/inputs')
for subject in subjects:
    inputs += m * [subject]
df = pd.DataFrame({'input': inputs})

# target column
df['target'] = [f'{i:04d}.bmp' for i in range(1, n+1)]

# hair column
hair = []
for color in colors:
    hair += len(colors) * [color]
df['hair'] = int(n/m) * hair

# shirt column
shirt = int(n / c) * colors
df['shirt'] = shirt

df.to_csv(f'processed/{phase}/metadata.csv', index=None)